<a href="https://colab.research.google.com/github/dswhaley/UFC-Fight-Prediction/blob/main/UFC_Project_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 500)
pd.set_option('display.expand_frame_repr', False)

path = kagglehub.dataset_download("mdabbert/ultimate-ufc-dataset")



In [ ]:
print(os.listdir(path))

df = pd.read_csv(path + '/ufc-master.csv')
df.head()


#Phase 1 - Cleaning
My end objective is given the data, I want to be able to predict whether the Red fighter or the Blue fighter will win the match.

Since I am trying to predict the winner of the fight, and not how the fight is won, there are some features that are unecessary. For example, I am not trying to predict how the fight will end, so I do not need the odds for DecOdds, SubOdds, or KOodds. I will keep Finish, because I think that the method of victory could matter, I want to look into whether or not you have a higher chance of winning if you are a striker or if you are a grappler and I need method of victory for that. However I do not need to know the exact submission someone uses, or how someone was knocked out. There are some submissions that are so few that I believe it would cause overfitting if I were to spread them out, so in addition to removing the odds, I will also remove the FinishDetails. I will also get rid of EmptyArena since this was something that happend in 2020 and is not relevant anymore.




In [ ]:

df = df.drop(columns=['RedDecOdds', 'BlueDecOdds', 'RSubOdds', 'BSubOdds', 'RKOOdds', 'BKOOdds', 'FinishDetails','EmptyArena'])
df.head()



After removing data that is not relevant to my problem, the next issue is the amount of different columns telling us how the fighter is ranked. The Fighters rank is certainly important. But it only really matters what there rank is in the weight class that the fight is occuring in. It is true that occasionally fighters move up weight classes, but this is pretty rare. However it did happen in 2024 when the #2 ranked Featherweight Max Holloway went up and fought the #2 ranked lightweight Justin Gaethje at Lightweight. If we threw out the other fighters rank completely if they were not ranked in that weightclass I think it would effect the weighting of ranked fighters verse unranked fighters. For example, Max Holloway won that fight, while he was unranked in the lightweight division, he was ranked 2nd in the Feather Weight division. In order to know if this is relevant or not I must test it.

First I want to gather some data, I am going to find the number of times that a fighter is ranked in another division and unranked in the division that they are fighting in. Then I am going to find out the number of times that the unranked fighter beats the ranked fighter. I will also find the number of times that unranked fighters who are not ranked in another division beat ranked fighters to give a comparison. I expect it to matter but I want to see the result of my tests first.


In [ ]:
rank_cols = [col for col in df.columns if 'rank' in col.lower()]
print(rank_cols)

I am going to store the rank of the fighters for that weight class, and if the fighter is not ranked in that weight class. I will check if the fighter is ranked in any weightclass. If they are I will store that in, OtherWeightClassRank. This way there can be weight added to offset the fact that they aren't ranked in the weightclass that they are fighting in.



In [ ]:
red_rank_cols = [
    'RWFlyweightRank', 'RWFeatherweightRank', 'RWStrawweightRank', 'RWBantamweightRank',
    'RHeavyweightRank', 'RLightHeavyweightRank', 'RMiddleweightRank', 'RWelterweightRank',
    'RLightweightRank', 'RFeatherweightRank', 'RBantamweightRank', 'RFlyweightRank', 'RMatchWCRank'
]

blue_rank_cols = [
    'BWFlyweightRank', 'BWFeatherweightRank', 'BWStrawweightRank', 'BWBantamweightRank',
    'BHeavyweightRank', 'BLightHeavyweightRank', 'BMiddleweightRank', 'BWelterweightRank',
    'BLightweightRank', 'BFeatherweightRank', 'BBantamweightRank', 'BFlyweightRank', 'BMatchWCRank'
]
def createdict(cols, row):
  wcRanks = {}
  for col in cols:
    wcRanks[col] = row[col]
  return wcRanks

def check_if_ranked(color, weightClass, wcranks):
  for key, value in wcranks.items():
    if pd.notna(value) and key != f'{color}{weightClass}Rank':
      return True
  return False


unranked_fighters_ranked_in_other_weightclass_fighting_ranked_opponents = 0
uranked_fighters_unranked_in_other_weightclassess_fighting_ranked_opponents = 0
case1UnrankedWins= 0
case1RankedWins = 0
case2UnrankedWins = 0
case2RankedWins = 0




for index, row in df.iterrows():
  weightClass = row['WeightClass']
  weightClass = weightClass.replace(" ", "").replace("'", "")

  if weightClass == 'CatchWeight' or 'Women' in weightClass: # There are no ranks in the df for women or catchweight fights
    continue
  red_dict = createdict(red_rank_cols, row)
  blue_dict = createdict(blue_rank_cols, row)


  if(pd.isna(row[f'B{weightClass}Rank'])) and pd.notna(row[f'R{weightClass}Rank']):
    if(check_if_ranked('B', weightClass, blue_dict)):
      unranked_fighters_ranked_in_other_weightclass_fighting_ranked_opponents += 1
      if(row['Winner'] == 'Red'):
        case1RankedWins += 1
      else:
        case1UnrankedWins += 1
    else:
      uranked_fighters_unranked_in_other_weightclassess_fighting_ranked_opponents += 1
      if(row['Winner'] == 'Red'):
        case2RankedWins += 1
      else:
        case2UnrankedWins += 1
  elif pd.notna(row[f'B{weightClass}Rank']) and pd.isna(row[f'R{weightClass}Rank']):
    if(check_if_ranked('R', weightClass, red_dict)):
      unranked_fighters_ranked_in_other_weightclass_fighting_ranked_opponents += 1
      if(row['Winner'] == 'Red'):
        case1UnrankedWins += 1
      else:
        case1RankedWins += 1
    else:
      uranked_fighters_unranked_in_other_weightclassess_fighting_ranked_opponents += 1
      if(row['Winner'] == 'Red'):
        case2UnrankedWins += 1
      else:
        case2RankedWins += 1

print(f'There are {unranked_fighters_ranked_in_other_weightclass_fighting_ranked_opponents} fighters fighting unranked in fights when they are ranked in other weight classes \n The unranked fighter has won {case1UnrankedWins} times. \nThis means that when a fighter ranked in another weightclass fights a fighter unranked in another weightclass they win {(case1UnrankedWins/unranked_fighters_ranked_in_other_weightclass_fighting_ranked_opponents)* 100}% of the time ')

print('\n\n\n')
print(f'There are {uranked_fighters_unranked_in_other_weightclassess_fighting_ranked_opponents} fighters fighting unranked in fights when they are unranked in other weight classes \nIn those cases, the unranked fighter won {case2UnrankedWins} times. \nThis means that when a fighter unranked fights a fighter unranked in another weightclass they win {(case2UnrankedWins/uranked_fighters_unranked_in_other_weightclassess_fighting_ranked_opponents) * 100}% of the time ')

Unfortunately this did not prove to be as big of a deal as I expected. It turns out that the times that this happens is a huge deal, so all of the UFC fans hear about it when it happens. But in reality it has only happened 50 times. Not only is it not as prevelant as I expected, but there isn't a very big difference in win percentage as the "unranked but ranked" fighter only wins 40% of the time compared to the other 37% of the time. To use this data would get into overfitting territory. Unfortunately...last night I did not test to see if this was predictive or not before I spent a good amount of time fighting with pandas and working on the code to make this happen. So I am going to leave the code here to show what would have happened.

In [ ]:
#Note for this code to work, It would need to be after the next code block, I am just leaving it here

def set_alternate_wc_rank(color: str, row):
  if(row[f'{color}WeightClassRank'] != 0):
    return 0

  if color == 'R':
    cols = red_rank_cols
  else:
    cols = blue_rank_cols

  wcRanks = createdict(cols, row)
  weightClass = row['WeightClass']
  weightClass = weightClass.replace(" ", "").replace("'", "")

  rank_value = wcRanks.get(f'{color}{weightClass}Rank')

  for key, value in wcRanks.items():
    if pd.notna(value) and key != f'{color}{weightClass}Rank':
      return value

  return 0


# df['BAlternateWeightClassRank'] = df.apply(lambda row: set_alternate_wc_rank('B', row), axis=1)
# df['RAlternateWeightClassRank'] = df.apply(lambda row: set_alternate_wc_rank('R', row), axis=1)


I want to make a change to the rankings. In the UFC the lower the rank is the better fighters are ranked 1-15 with the champion being ranked 0. This will make it hard to attatch weights later as it is hard to know what to do with unranked fighters. Its very easy though if I were to flip it and make the champion ranked 16 and all unranked fighters ranked 0. So this is what I am going to do.

In [ ]:
def invert_rank(rank):
    if pd.isna(rank):
        return 0
    else:
        return 16 - rank

all_rank_cols = red_rank_cols + blue_rank_cols
for col in all_rank_cols:
    df[col] = df[col].apply(invert_rank)


Now that I have fixed the rankings, I can look more closely at the ranking information. Thankfully everything I did last night was not a waste. While the alternate weightclass rank is not useful it is always a good idea to get more data by combining weightclasses. By storing the ranks of fighters in the weightcvlass of the fight I have a lot of data on ranked fighters perform against each other.

In [ ]:

# I do not have much to any experience with pandas, so I used ChatGPT to get a better understanding for how to manipulate dfs. I didn't know if I should iterate over  the whole df. It told me to use '.apply()'. Other than that I created the functions myself besides using chatgpt like google to learn about pandas. I did not have it generate any code.






def set_wc_rank(color: str, row):
  if color == 'R':
    cols = red_rank_cols
  else:
    cols = blue_rank_cols
  wcRanks = createdict(cols, row)
  weightClass = row['WeightClass']
  weightClass = weightClass.replace(" ", "").replace("'", "")
  rank_value = wcRanks.get(f'{color}{weightClass}Rank')

  if pd.notna(rank_value):
    return wcRanks.get(f'{color}{weightClass}Rank')

  return 0





red_rank_cols = [
      'RWFlyweightRank',
      'RWFeatherweightRank',
      'RWStrawweightRank',
      'RWBantamweightRank',
      'RHeavyweightRank',
      'RLightHeavyweightRank',
      'RMiddleweightRank',
      'RWelterweightRank',
      'RLightweightRank',
      'RFeatherweightRank',
      'RBantamweightRank',
      'RFlyweightRank',
      'RMatchWCRank'
]

blue_rank_cols = [
    'BWFlyweightRank',
    'BWFeatherweightRank',
    'BWStrawweightRank',
    'BWBantamweightRank',
    'BHeavyweightRank',
    'BLightHeavyweightRank',
    'BMiddleweightRank',
    'BWelterweightRank',
    'BLightweightRank',
    'BFeatherweightRank',
    'BBantamweightRank',
    'BFlyweightRank',
    'BMatchWCRank'
  ]


df['RWeightClassRank'] = df.apply(lambda row: set_wc_rank('R', row), axis=1)
df['BWeightClassRank'] = df.apply(lambda row: set_wc_rank('B', row), axis=1)


df.head()

Now that I have done this, I no longer need all of the individual rankings for each weightclass as it is already represented in weightclass and WeightClassRank, so I can remove the extra columns.

In [ ]:
df = df.drop(columns=all_rank_cols)
df.head()

Now that that has been taken care of there are some other cleaning things that I want to take care of. Starting with Winning. Right now Winner gives me Red and Blue, but I don't find this particularly helpful as I can't graph it and I cannot predict it as easily. So I am going to make winner be 1 when the Red fighter wins and winner be 0 when the blue fighter wins. In other words, I am trying to predict if the Red Fighter will win the fight, and if I predict that he won't win I am really predicting that the blue fighter will win the fitht.

In [ ]:
df['Winner'] = df['Winner'].apply(lambda x: 1 if x == 'Red' else 0)
df.head()

Another ranking thing I need to change is the Pound for Pound rankings. I need to invert the rankings like I did earlier as well as set all unranked fighters to 0. I didn't remove this column because pfp rankings are not represented anywhere else in my dataset.

In [ ]:
df['BPFPRank'] = df['BPFPRank'].apply(invert_rank)
df['RPFPRank'] = df['RPFPRank'].apply(invert_rank)

df['BPFPRank'] = df['BPFPRank'].apply(lambda x: 0 if pd.isna(x) else x)
df['RPFPRank'] = df['RPFPRank'].apply(lambda x: 0 if pd.isna(x) else x)


df.head()

Instead of Having BetterRank Display Red or Blue as the outcome, I am going to remove that column and make it so that when betterRank is Red, the new Column, RedBetterRank = 1, and when betterRank is Blue, the new column BlueBetterRank is 1.




In [ ]:
df['RBetterRank'] = df['BetterRank'].apply(lambda x: 1 if x == 'Red' else 0)
df['BBetterRank'] = df['BetterRank'].apply(lambda x: 1 if x == 'Blue' else 0)
df = df.drop(columns=['BetterRank'])
df.head()

Another small thing that I need to take care of is making the title bout numerical binary. This is fairly easy to take care of.


In [ ]:
df['TitleBout'] = df['TitleBout'].apply(lambda x: 1 if x == True else 0)
df.head()

Instead of having Gender have Male and Female options, I am going to make two seperate columns for them and set them to 1 or 0.

In [ ]:
df['GenderMale'] = df['Gender'].apply(lambda x: 1 if x == 'MALE' else 0)
df['GenderFemale'] = df['Gender'].apply(lambda x: 1 if x == 'FEMALE' else 0)

df = df.drop(columns=['Gender'])
df.head()

I am now going to split BlueStance and BlueStance into RSouthpaw, ROrthodox, BSouthpaw, BOthodox. While I am doing this, I am also going to add a colum called differentStance because I want to run some experiements on that in the data exploration phase.

In [ ]:
df['RSouthpaw'] = df['RedStance'].apply(lambda x: 1 if x == 'Southpaw' else 0)
df['ROthodox'] = df['RedStance'].apply(lambda x: 1 if x == 'Orthodox' else 0)
df['BSouthpaw'] = df['BlueStance'].apply(lambda x: 1 if x == 'Southpaw' else 0)
df['BOthodox'] = df['BlueStance'].apply(lambda x: 1 if x == 'Orthodox' else 0)

df['differentStance'] = df.apply(lambda row: 1 if row['RedStance'] != row['BlueStance'] else 0, axis=1)

df = df.drop(columns=['RedStance', 'BlueStance'])
df.head()


Now I am going to split the Finish column, I am going to make a colunn called, U-Dec, S-Dec, Sub, KO/TKO

In [ ]:
df['UDec'] = df['Finish'].apply(lambda x: 1 if x == 'U-DEC' else 0)
df['SDec'] = df['Finish'].apply(lambda x: 1 if x == 'S-DEC' else 0)
df['Sub'] = df['Finish'].apply(lambda x: 1 if x == 'SUB' else 0)
df['KO/TKO'] = df['Finish'].apply(lambda x: 1 if x == 'KO/TKO' else 0)

df = df.drop(columns=['Finish'])
df.head()

I am going to change weightclass from being categorical data to being numerical by assigning each weightclass to different values. Weightclasses are extremely important as fights tend to be different at different weight classes. Without giving numerical data for the weightclass, I would not be able to easily make a model for this data.

In [ ]:
df['WeightClass'].unique()



In [ ]:
weight_mapping = {
    "Women's Strawweight": 1,
    "Women's Flyweight": 2,
    "Flyweight": 3,
    "Bantamweight": 4,
    "Women's Bantamweight": 5,
    "Featherweight": 6,
    "Women's Featherweight": 7,
    "Lightweight": 8,
    "Welterweight": 9,
    "Middleweight": 10,
    "Light Heavyweight": 11,
    "Heavyweight": 12,
    "Catch Weight": 13
}

# Apply mapping to create numerical column
df['weight_class_num'] = df['WeightClass'].map(weight_mapping)


In [ ]:
df.head()

#Phase 2 - Data Exploration
At this point the data is fairly clean, there is no more NaN values in the data, and I have removed all of the unnecessary columns. There is of course more data cleaning that could be done, and hopefully if I have time I can get to it, but since a large portion of my time on this project so far has been spend on data cleaning it is time for me to start learning about correlation in my data.



##Last Fight Knocked Out

There is a theory that fighters tend to do worse in the fight after they have been knocked out. I want to test if this is true or not. To do that I need to add two columns, RLastFightWasKOed and BLastFightWasKOed, I then need to go through the data and find out if this is true or not.

In [ ]:

def koed_last_fight(fighter, row):
  fighter_fights = df[
          (df['RedFighter'] == fighter) |
          (df['BlueFighter'] == fighter)
      ]
  last_fight = fighter_fights.iloc[0]
  if(last_fight['RedFighter'] == fighter and last_fight['Winner'] == 0 and last_fight['KO/TKO'] == 1):
    return 1
  elif(last_fight['BlueFighter'] == fighter and last_fight['Winner'] == 1 and last_fight['KO/TKO'] == 1):
    return 1

  return 0

df['BLastFightWasKOed'] = df.apply(lambda row: koed_last_fight(row['BlueFighter'], row), axis=1)
df['RLastFightWasKOed'] = df.apply(lambda row: koed_last_fight(row['RedFighter'], row), axis=1)


In [ ]:
fighter_fights = df[
          (df['RedFighter'] == 'Conor McGregor') |
          (df['BlueFighter'] == 'Conor McGregor')
      ]
fighter_fights.head(2)

Connor McGreggor was knocked out in both of his last two fights, and this correctly added the 1 to fighter was knocked out showing that this code did what it was supposed to do. Now it is time to see if there is a correlation between last fight ending in a knockout and the fighter losing the next fight. To do that I need to find how fighters did in their next fight after this occured.



In [ ]:
fighters_koed_in_last_fight = pd.concat([
    df[df['RLastFightWasKOed'] == True][['RedFighter', 'Winner', 'RLastFightWasKOed', 'RedAge']].assign(Color='Red'),
    df[df['BLastFightWasKOed'] == True][['BlueFighter', 'Winner', 'BLastFightWasKOed', 'BlueAge']].assign(Color='Blue')
], ignore_index=True) #This was made with the help of AI, I didn't just ask them to do it for me, it was a back and forth process

num_fighters_koed_in_last_fight = len(fighters_koed_in_last_fight)
print(f"Number of fighters who were knocked out in their last fight: {num_fighters_koed_in_last_fight}")

num_losses = 0
for index, row in fighters_koed_in_last_fight.iterrows():
  if(row['Color'] == 'Red' and row['Winner'] == 0):
    num_losses += 1
  elif(row['Color'] == 'Blue' and row['Winner'] == 1):
    num_losses += 1

print(f"Number of losses: {num_losses}")


print(f'So {(num_losses / num_fighters_koed_in_last_fight) * 100} of fighters who were knocked out in their last fight, lost their next fight.')

This is very relevant data there are over 3000 fighters who fight after they have been knocked out, and in those fights they lose 55% of the time. Therefore it is absolutely worth having in consideration when trying to determine if a fighter will win their next fight.


After doing that analyses, I decided I wanted to see how age influenced that. I didn't know how to go to create those graphs, so I used AI.I gave it the code above and asked them to plot the loss percentage and age of the fighters who were knocked out. I also asked them to graph the counts of the fighters ages who were knocked out.  This is what it resulted in. I had to change df['RLastFightWasKOed'] == True to == 1 because the AI didn't understand how the dataframe was setup.

In [ ]:
# Combine fighters who were KOed in their last fight
fighters_koed_in_last_fight = pd.concat([
    df[df['RLastFightWasKOed'] == 1][['RedFighter', 'Winner', 'RLastFightWasKOed', 'RedAge']].assign(Color='Red'),
    df[df['BLastFightWasKOed'] == 1][['BlueFighter', 'Winner', 'BLastFightWasKOed', 'BlueAge']].assign(Color='Blue')
], ignore_index=True)

# Rename age column for consistency and ensure it's a single column
fighters_koed_in_last_fight['Age'] = fighters_koed_in_last_fight['RedAge'].fillna(fighters_koed_in_last_fight['BlueAge'])
fighters_koed_in_last_fight = fighters_koed_in_last_fight.drop(columns=['RedAge', 'BlueAge'])


# Add a column 'LostNextFight' (1 if lost, 0 if won)
def determine_loss(row):
    if row['Color'] == 'Red' and row['Winner'] == 0:
        return 1
    elif row['Color'] == 'Blue' and row['Winner'] == 1:
        return 1
    else:
        return 0

fighters_koed_in_last_fight['LostNextFight'] = fighters_koed_in_last_fight.apply(determine_loss, axis=1)

# Group by age: calculate loss percentage and number of fighters
age_loss_stats = fighters_koed_in_last_fight.groupby('Age').agg(
    LossPercentage=('LostNextFight', lambda x: x.mean() * 100), # Calculate percentage and rename
    NumFighters=('LostNextFight', 'size')  # Count number of fighters using size
).reset_index()



# ------------------------
# Graph 1: Loss Percentage vs Age
# ------------------------
plt.figure(figsize=(10,6))
sns.lineplot(data=age_loss_stats, x='Age', y='LossPercentage', marker='o', color='red')
plt.title('Percentage of Fighters Losing Next Fight vs Age')
plt.xlabel('Age')
plt.ylabel('Loss Percentage (%)')
plt.grid(True)
plt.show()

# ------------------------
# Graph 2: Number of Fighters KOed in Last Fight vs Age
# ------------------------
plt.figure(figsize=(10,6))
sns.barplot(data=age_loss_stats, x='Age', y='NumFighters', color='blue')
plt.title('Number of Fighters KOed in Last Fight vs Age')
plt.xlabel('Age')
plt.ylabel('Number of Fighters')
plt.grid(True)
plt.show()

This is interesting and shows exactly what I was expecting, While the average is 55%, older fighters are more likely to get knocked out in fights after they have been knocked out and younger fighters tend to be less likely to lose, though only slightly. It could be very useful to take age combined with if they were knocked out in their last fight when trying to predict if a fighter will win a fight. Fighters are defintely more likely to lose in fights after they have been knocked out, so the fan theory is correct.

##Grapplers and Strikers

In the UFC fighters are typically classified as grapplers or strikers, I want to determine what a grappler is and what a striker is and then see what the win percentage is for each of them. To do that I need to first figure out what it means to be a Grappler and what it means to be a striker.


My inclination is that Fighters who have higher Submission Attempts, Submission Wins, TakeDown Attempts, and Takedowns will be considered Grapplers, and that Fighters with higher significant strikes, significant strike% and wins by KO will be considered strikers. To determine the best threshold to classify them, I need to find the averages for each metric.

In [ ]:
df.head(1)

In [ ]:

grappler_metrics = {
    'AvgSubAtt': pd.concat([df['RedAvgSubAtt'], df['BlueAvgSubAtt']]),
    'WinsBySubmission': pd.concat([df['RedWinsBySubmission'], df['BlueWinsBySubmission']]),
    'AvgTDLanded': pd.concat([df['RedAvgTDLanded'], df['BlueAvgTDLanded']]),
    'AvgTDPct': pd.concat([df['RedAvgTDPct'], df['BlueAvgTDPct']])
}

striker_metrics = {
    'AvgSigStrLanded': pd.concat([df['RedAvgSigStrLanded'], df['BlueAvgSigStrLanded']]),
    'AvgSigStrPct': pd.concat([df['RedAvgSigStrPct'], df['BlueAvgSigStrPct']]),
    'WinsByKO': pd.concat([df['RedWinsByKO'], df['BlueWinsByKO']])
}



I used AI to help me generate the code above, I was struggling to get concat to work. I understand what I need to do after I get the red and the blue combined, but combining them was much more difficult, I asked it for help and it gave me the code above. Now I am going to find the averages for each metric and evaluate how to classify someone as a grappler and a striker.

In [ ]:
for metric, values in grappler_metrics.items():
    average = values.mean()
    print(f"Average for {metric}: {average}")
for metric, values in striker_metrics.items():
    average = values.mean()
    print(f"Average for {metric}: {average}")

This was lower than I expected. I want to compute these again but only take into consideration times when the fighter has fought more than 5 times.

In [ ]:
df['RedTotalFights'] = df['RedWins'] + df['RedLosses'] + df['RedDraws']
df['BlueTotalFights'] = df['BlueWins'] + df['BlueLosses'] + df['BlueDraws']

red_mask = df['RedTotalFights'] >= 5
blue_mask = df['BlueTotalFights'] >= 5


grappler_metrics = {
    'AvgSubAtt': pd.concat([df.loc[red_mask, 'RedAvgSubAtt'], df.loc[blue_mask, 'BlueAvgSubAtt']]),
    'WinsBySubmission': pd.concat([df.loc[red_mask, 'RedWinsBySubmission'], df.loc[blue_mask, 'BlueWinsBySubmission']]),
    'AvgTDLanded': pd.concat([df.loc[red_mask, 'RedAvgTDLanded'], df.loc[blue_mask, 'BlueAvgTDLanded']]),
    'AvgTDPct': pd.concat([df.loc[red_mask, 'RedAvgTDPct'], df.loc[blue_mask, 'BlueAvgTDPct']])
}

striker_metrics = {
    'AvgSigStrLanded': pd.concat([df.loc[red_mask, 'RedAvgSigStrLanded'], df.loc[blue_mask, 'BlueAvgSigStrLanded']]),
    'AvgSigStrPct': pd.concat([df.loc[red_mask, 'RedAvgSigStrPct'], df.loc[blue_mask, 'BlueAvgSigStrPct']]),
    'WinsByKO': pd.concat([df.loc[red_mask, 'RedWinsByKO'], df.loc[blue_mask, 'BlueWinsByKO']])
}

In [ ]:
for metric, values in grappler_metrics.items():
    average = values.mean()
    print(f"Average for {metric}: {average}")
for metric, values in striker_metrics.items():
    average = values.mean()
    print(f"Average for {metric}: {average}")

It turns out that they were what they should have been and I was wrong in thinking that they were too low. Now it is time to classify them.

I wanted to classify fighters as just Grapplers and Strikers but the more I thought about it, that wouldn't give me the data that I desired. I think that it puts too high of an emphasis on takedowns for grappling. Instead I am going to break it into Wrestler, Grappler, and Striker. I will allow fighters who are ballanced to be put in more than one category if they are above average in more than one so that they can still get the credit for being a good grappler and a good wrestler if they are profficient in both. However there are fighters like Charles Oliveira who are very good grapplers with lots of submissions and lots of submission attempts, but they don't shoot very much. It would be criminal not to classify fighters like that as grapplers.


In [ ]:
grappler_metrics = {
    'AvgSubAtt': pd.concat([df['RedAvgSubAtt'], df['BlueAvgSubAtt']]).mean(),
    'WinsBySubmission': pd.concat([df['RedWinsBySubmission'], df['BlueWinsBySubmission']]).mean(),
}

wrestler_metrics = {
    'AvgTDLanded': pd.concat([df['RedAvgTDLanded'], df['BlueAvgTDLanded']]).mean(),
    'AvgTDPct': pd.concat([df['RedAvgTDPct'], df['BlueAvgTDPct']]).mean()
}

striker_metrics = {
    'AvgSigStrLanded': pd.concat([df['RedAvgSigStrLanded'], df['BlueAvgSigStrLanded']]).mean(),
    'AvgSigStrPct': pd.concat([df['RedAvgSigStrPct'], df['BlueAvgSigStrPct']]).mean(),
    'WinsByKO': pd.concat([df['RedWinsByKO'], df['BlueWinsByKO']]).mean()
}

for metric, values in grappler_metrics.items():
    print(f"Average for {metric}: {values}")
for metric, values in striker_metrics.items():
    print(f"Average for {metric}: {values}")
for metric, values in wrestler_metrics.items():
    print(f"Average for {metric}: {values}")

df['RedIsGrappler'] = ((df['RedAvgSubAtt'] > grappler_metrics['AvgSubAtt']) & (df['RedWinsBySubmission'] > grappler_metrics['WinsBySubmission'])).astype(int)
df['BlueIsGrappler'] = ((df['BlueAvgSubAtt'] > grappler_metrics['AvgSubAtt']) & (df['BlueWinsBySubmission'] > grappler_metrics['WinsBySubmission'])).astype(int)
df['RedIsWrestler'] = ((df['RedAvgTDLanded'] > wrestler_metrics['AvgTDLanded']) & (df['RedAvgTDPct'] > wrestler_metrics['AvgTDPct'])).astype(int)
df['BlueIsWrestler'] = ((df['BlueAvgTDLanded'] > wrestler_metrics['AvgTDLanded']) & (df['BlueAvgTDPct'] > wrestler_metrics['AvgTDPct'])).astype(int)
df['RedIsStriker'] = ((df['RedAvgSigStrLanded'] > striker_metrics['AvgSigStrLanded']) & (df['RedAvgSigStrPct'] > striker_metrics['AvgSigStrPct']) & (df['RedWinsByKO'] > striker_metrics['WinsByKO'])).astype(int)
df['BlueIsStriker'] = ((df['BlueAvgSigStrLanded'] > striker_metrics['AvgSigStrLanded']) & (df['BlueAvgSigStrPct'] > striker_metrics['AvgSigStrPct']) & (df['BlueWinsByKO'] > striker_metrics['WinsByKO'])).astype(int)

Now that we have them classified, lets look at the population of each one.

In [ ]:
all_fighters = pd.DataFrame({
    'Grappler': pd.concat([df['RedIsGrappler'], df['BlueIsGrappler']], ignore_index=True),
    'Wrestler': pd.concat([df['RedIsWrestler'], df['BlueIsWrestler']], ignore_index=True),
    'Striker': pd.concat([df['RedIsStriker'], df['BlueIsStriker']], ignore_index=True)
})

style_counts = all_fighters.sum()
style_counts.plot(kind='bar', color=['skyblue','orange','green'])
plt.ylabel('Number of Fighters')
plt.title('Number of Fighters by Style (All Combined)')
plt.show()

Suprisingly there are only 500 strikers, I would not have expected that. Now I want to see the win percentage for each group.

In [ ]:

all_fighters = pd.DataFrame({
    'Style_Grappler': pd.concat([df['RedIsGrappler'], df['BlueIsGrappler']], ignore_index=True),
    'Style_Wrestler': pd.concat([df['RedIsWrestler'], df['BlueIsWrestler']], ignore_index=True),
    'Style_Striker': pd.concat([df['RedIsStriker'], df['BlueIsStriker']], ignore_index=True),
    'Won': pd.concat([df['Winner'], 1 - df['Winner']], ignore_index=True)
})

grappler_win_pct = all_fighters.loc[all_fighters['Style_Grappler']==1, 'Won'].mean() * 100
wrestler_win_pct = all_fighters.loc[all_fighters['Style_Wrestler']==1, 'Won'].mean() * 100
striker_win_pct = all_fighters.loc[all_fighters['Style_Striker']==1, 'Won'].mean() * 100

print(f"Grappler Win %: {grappler_win_pct:.1f}%")
print(f"Wrestler Win %: {wrestler_win_pct:.1f}%")
print(f"Striker Win %: {striker_win_pct:.1f}%")


I was running short on time so I had AI help me find the win percentages, but I definitely could've done it myself by fighting with it.

We do learn valuable information here. Wrestlers win more than either group of people and grapplers beat strikers. Each of these is above 50% although barely, both of them are above 52.4% which is regarded as the percentage needed to be profitable in sportsbetting on -110 odds (https://leans.ai/betting/how-to/what-percent-of-bets-to-win-to-be-profitable/) So even though it might seem negligible, It is positve, the only one under it is striker. Even though it seems like a small difference, it actually is quite a big deal.

Now that I have figured out their win percentage, I want to see how each group does against each other. I am going to find the win percentage for wrestlers against strikers and wrestlers against grapplers, and strikers against grapplers. (I will then use the inverse to compute the values for the others)

In [ ]:
all_fights = pd.DataFrame({
    'Red_Grappler': df['RedIsGrappler'],
    'Red_Wrestler': df['RedIsWrestler'],
    'Red_Striker': df['RedIsStriker'],
    'Blue_Grappler': df['BlueIsGrappler'],
    'Blue_Wrestler': df['BlueIsWrestler'],
    'Blue_Striker': df['BlueIsStriker'],
    'Winner': df['Winner']  # 1 = Red, 0 = Blue
})

def win_percentage(style1_red_col, style1_blue_col, style2_red_col, style2_blue_col):
    """
    Compute win % for style1 vs style2.
    style1_red_col: Red fighter column for style1
    style1_blue_col: Blue fighter column for style1
    style2_red_col: Red fighter column for style2
    style2_blue_col: Blue fighter column for style2
    """
    # Fights where style1 is Red and style2 is Blue
    mask_red = (all_fights[style1_red_col] == 1) & (all_fights[style2_blue_col] == 1)
    red_wins = all_fights.loc[mask_red, 'Winner'].mean()  # Red = style1

    # Fights where style1 is Blue and style2 is Red
    mask_blue = (all_fights[style1_blue_col] == 1) & (all_fights[style2_red_col] == 1)
    blue_wins = (1 - all_fights.loc[mask_blue, 'Winner']).mean()  # Blue = style1

    # Combine
    combined_win_pct = pd.concat([all_fights.loc[mask_red, 'Winner'],
                                  1 - all_fights.loc[mask_blue, 'Winner']]).mean() * 100
    return combined_win_pct

# Wrestlers vs Strikers
wrestler_vs_striker = win_percentage('Red_Wrestler', 'Blue_Wrestler', 'Red_Striker', 'Blue_Striker')

# Wrestlers vs Grapplers
wrestler_vs_grappler = win_percentage('Red_Wrestler', 'Blue_Wrestler', 'Red_Grappler', 'Blue_Grappler')

# Strikers vs Grapplers
striker_vs_grappler = win_percentage('Red_Striker', 'Blue_Striker', 'Red_Grappler', 'Blue_Grappler')

# Original calculations
print(f"Wrestlers vs Strikers: {wrestler_vs_striker:.1f}%")
print(f"Wrestlers vs Grapplers: {wrestler_vs_grappler:.1f}%")
print(f"Strikers vs Grapplers: {striker_vs_grappler:.1f}%")

striker_vs_wrestler = 100 - wrestler_vs_striker
grappler_vs_wrestler = 100 - wrestler_vs_grappler
grappler_vs_striker = 100 - striker_vs_grappler

print(f"Strikers vs Wrestlers: {striker_vs_wrestler:.1f}%")
print(f"Grapplers vs Wrestlers: {grappler_vs_wrestler:.1f}%")
print(f"Grapplers vs Strikers: {grappler_vs_striker:.1f}%")

I used AI for this aswell. Again I completely understand the code, it was just faster than writing it myself. I had my code examples I could give the AI mixed with very specific prompts it was able to create results that worked right away.

I find these results fasinating. Striking is definitely the fan favorite fighting style, but the data shows that wrestling is the most successfull style of fighting against every other form. Its very colse with Grapplers, only a slight edge. But it is definitiely an edge. It's very intruiging how successful it is against strikers. Strikers do not fair well against grappling or wrestling. This information will be extremely helpful when creating a model because it shows that if someone has a good ground game, they will probably win the fight. I must admit as a Grappler/Wrestler I find it very satisfying knowing that my style of fighting is superior.

I strongly believe that these new features will be very significant when trying to predict winners.

I genuienly had no idea which style would win, the results were very interesting and I belive that they will be very useful when modeling.

#Finding Correlation
Now I want to find correlations between my features and fighters winning fights. Right now that is pretty hard to do since every row contains both the information for the winner and the information for the loser on one line. My goal now is to make a new df for identifying correlations. So that I can more easily plot information and identify correlations. In heindsight this is probably something I should have done much earlier but I did not think about it until I talked to some of the other people about their projects.

In order to make this dataframe I will drop all of the information for the actual fight, so if the result is a KO, for this example, I do not care because I already have if the fighters was knocked out last fight. What I want here is all of the fighters stats, so that I can find correlation between the stats and to find correaltion with winning fights.

I will then make a new table that is fighter's name and date, and then instead of having red and blue as the features it will just be the features as well as the outcome. Note that in the origional df, all of the "diff" features were positive if it was in Blues favor and negative if it was in reds. For that reason I multiplied all of Red's differentials by -1 in order to see what the differentials actually were.

In [ ]:
df[["BlueWins", "RedWins", "WinDif"]].head()

In [ ]:
corr_df =  df.copy()


differentials = [c for c in df.columns if "Dif" in c]
red_cols = [c for c in df.columns if not c.startswith("Blue") and not c.startswith("B")]
blue_cols = [c for c in df.columns if not c.startswith("Red") and (not c.startswith("R"))]

red_df = df[red_cols].copy()
red_df["Win"] = (df["Winner"] == 1).astype(int)
red_df["Loss"] = (df["Winner"] == 0).astype(int)
red_df["isred"] = 1
red_df["isblue"] = 0


print(differentials)

blue_df = df[blue_cols].copy()
blue_df["Win"] = (df["Winner"] == 0).astype(int)
blue_df["Loss"] = (df["Winner"] == 1).astype(int)
blue_df["isred"] = 0
blue_df["isblue"] = 1


def clean_red_name(c):
    if c.startswith("Red"):
        return c.replace("Red", "", 1)
    elif c.startswith("R") and c != "ReachDif":
        return c[1:]
    return c

def clean_blue_name(c):
    if c.startswith("Blue"):
        return c.replace("Blue", "", 1)
    elif c.startswith("B"):
        return c[1:]
    return c

red_df.columns = [clean_red_name(c) for c in red_df.columns]
blue_df.columns = [clean_blue_name(c) for c in blue_df.columns]


for c in differentials:
    red_df[c] = df[c] * -1


corr_df = pd.concat([red_df, blue_df], ignore_index=True)

corr_df.head()


Now I want to drop the row if a fighter has no fights in the ufc and has then no fight statistics. I also do not need the fighters name, wieghtclass, Country, or Location since that is not numerical and therefore is not plottable. I created a value already to reperesent weight class numerically.

I also don't need how the fight ended anymore as that is fight results that could be useful in predicting how fights will end, but since we have the fighters stats before the fight already, I do not need the outcome of their last fight in order to predict a win. Since I made a new win column, I no longer need my winner column as I expect it to always be a wash.


In [ ]:
no_fights_mask = (corr_df["Wins"] == 0) & (corr_df["Losses"] == 0) & (corr_df["Draws"] == 0)

corr_df = corr_df.drop(corr_df[no_fights_mask].index)
corr_df = corr_df.drop(columns=["Fighter", "WeightClass", "Country", "Location", "ExpectedValue", "UDec", "SDec", "Sub", "KO/TKO", "Winner"])

corr_df.head()

Now that I have all of my data clean, I am going to start looking for correlations. To do that I had gemini create a pair plot for me of my data.

---



---



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Drop non-numeric columns before calculating correlation
corr_df_numeric = corr_df.drop(columns=['Date', 'FinishRoundTime'])

plt.figure(figsize=(35, 30))
sns.heatmap(corr_df_numeric.corr(), cmap='coolwarm', center=0, annot=False, square=True)
plt.title('Correlation Matrix for Fighter Stats', fontsize=16)
plt.xticks(rotation=90, fontsize=25)
plt.yticks(rotation=0, fontsize=25)
plt.tight_layout()
plt.show()

The things that I learned from this were very interesting. There are a lot of different things that I could tell you about the correlation matrix so I will tell you the most significant correlations that I noticed as well as the most surprising correlations that I noticed.

The first thing that I noticed is that Height, Weight, and Reach are heavily correalted. This makes a lot of sense, but their heavy correlation makes me thing that maybe they should be binned later. I am hesitent to bin the weight as weight is pretty tied in to weigtclass, so

Another Interesting thing that I found was that a fighters weightclass positvely correlates with the number of KOs that a fighter has. This makes logical sense on a physics standpoint as heavyier fighters will be able to generate more force with their punches.

On the topic of weight, height, and reach, there tends to be a negative correlation between size and WinsByDecisions. This also makes sense for the same reason that bigger fighters tend to have more wins by KO.

One thing that became very clear when looking at the correlation matrix is that the red and blue assignments are not random. Red fighters tend to be the more seasoned fighters. This is shown by the fact that Red Fighters tend to be, more highly ranked, Win more, Have more finishes, fought more, had more title fights. Basically, Red Fighters seem to be better in almost every positive category.

There is a positive differential between age and odds. Note that a positive correlation with odds is not a good thing as in sportsbetting, + odds indicates that the sportsbook does not belive that fighter will win.

The correlation I found the most surprising by far was that age correlates with a fighters reach, weight, and height. I understand why it would correlate with a fighters weight given that older fighters tend to struggle to make weight, but I find it shocking that it corresponds with height and reach unless. My onle explanation is that it would be easier to have longer carreers as a bigger fighter...But that is not what I would have expected.


Those were the general correlation that I found the most interesting. Now I want to look at specific features correlations. I had Gemini create me a method that will give the correlation matrix for a given feature and display it in order of magintude. So highly positive and negative correlations will appear at the top, but lower correlations will appear at the bottom.

In [ ]:
def individual_correlation(feature):

  correlations = corr_df_numeric.corr()[feature]

  # Sort by absolute value, descending
  correlations_sorted = correlations.reindex(correlations.abs().sort_values(ascending=False).index)

  # Print the correlations in a readable format
  print(correlations_sorted)

  # Plot for easier visualization
  plt.figure(figsize=(8, len(correlations_sorted)/2))
  sns.barplot(x=correlations_sorted.values, y=correlations_sorted.index, palette='coolwarm')
  plt.title("Feature Correlation with {feature} (sorted by strength)")
  plt.xlabel(f"Correlation with {feature}")
  plt.ylabel("Feature")
  plt.show()

I am only going to look at the correlation for a couple features for times sake as I want to get to pair plots and binning. As I mentioned earlier, I am curious to the validity in the claim that being a southpaw is an advantage. So I will first compare the correlation for southpaws and Orthodox, and then I will look at the correlations for different stance.

In [ ]:
individual_correlation("Southpaw")

In [ ]:
individual_correlation("Othodox")

In [ ]:
individual_correlation("differentStance")

This data confirms the Southpaw Theory as there is a positve correlation between winning and being a southpaw (Albiet small) and a negative correlation with being a Orthodox.

Now it's time to look at what is probably the most important correlation. How do features correlate with Win? We will look at the graph and then identify some of the most significant features.

In [ ]:
correlations = corr_df_numeric.corr()['Win']

# Sort by absolute value, descending
correlations_sorted = correlations.reindex(correlations.abs().sort_values(ascending=False).index)

# Print the correlations in a readable format
print(correlations_sorted)

# Plot for easier visualization
plt.figure(figsize=(8, len(correlations_sorted)/2))
sns.barplot(x=correlations_sorted.values, y=correlations_sorted.index, palette='coolwarm')
plt.title("Feature Correlation with Win (sorted by strength)")
plt.xlabel("Correlation with Win")
plt.ylabel("Feature")
plt.show()

Unshockingly, the strongest correlation is with odds. It turns out that Vegas is good at what they do and they tend to get the odds correct. And Red fighters tend to win more aswell as they tend to lead in every positive fighting category. Another unsurprisinging correlation is with the win streak difference, it is no surprise that the fighter with the higher winstreak will probably win the fight.

Some of the most interesting and surprising correlations are with the following:

Age - Age has a negative correlation with winning showing that Father Time does always win in the end.

AvgTDDiff - This confirms what we saw earlier when we grouped fighters by wrestler, grappler, and striker, but wrestling is a very good strategy in the ufc. The fighters who are able to bring the fight to the ground on their own terms tend to win the fights as it has the highest performanced base win correlation. AvgTDDiff doubles the next catefory AvgSigStrPct in win correlation.It makes sense then why that even though to most it can cause for an unentertaining fight, more and more fighters are becoming wrestlers.

After that there is nothing particulalry unsuprising, fighters with higher winnstreaks tend to win more than fighters with lower ones, fighters who have high averages in stats tend to win. And as mentioned earlier, fighters who have are wrestlers and grapplers tend to win, and fighters who were knocked out in their last fight tend to lose.

The only other particularly interesting thing I found was that how many fights you have won has almost no effect on how a fighter does. It has a -.006 correlation, but the number of losses you have does matter as it has a -.07 correlation with win making it considerably more relevant.

Now that we have looked at the correlations, lets look at some pair plots to get a better understanding of what needs to be binned.

In [ ]:
# %matplotlib inline
# import seaborn as sns
# sns.pairplot(corr_df_numeric, hue='Win', height=1.5, corner=True);
# #sns.pairplot(corr_df_numeric, hue='Win', palette={0: 'red', 1: 'green'}, height=1.5)


This code never ran as the the table was far to large. Instead of running it on every feature, I will only run it on the features that had the highest correlation with Win, I need to shrink the table somehow, and that is what make sthe most sense to me.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select a subset of features for the pair plot that showed high correlation with 'Win'
selected_features = correlations_sorted.index[:10].tolist() # Get the index (feature names) and convert to a list

# Create a subset DataFrame
pair_plot_df = corr_df_numeric[selected_features].copy()

# Generate the pair plot
sns.pairplot(pair_plot_df,
             hue='Win',
             palette={0: 'red', 1: 'green'},
             diag_kind='kde',
             height=2)

plt.suptitle("Pair Plot of Selected Features Correlated with Win", y=1.02)
plt.show()

After looking at this, I will try again and this time I will not grab any binary values because they are not interesting on pair plots/

In [ ]:
selected_features = [
    'Win',                  # binary target
    'Odds',                 # -0.363
    'WinStreakDif',         # 0.134
    'AvgTDDif',             # 0.125
    'Age',                  # -0.114
    'LongestWinStreakDif',  # 0.088
    'AvgTDLanded',          # 0.082
    'CurrentWinStreak',     # 0.076
    'BetterRank'            # 0.075
]


pair_plot_df = corr_df_numeric[selected_features].copy()

# Generate the pair plot
sns.pairplot(pair_plot_df,
             hue='Win',
             palette={0: 'red', 1: 'green'},
             diag_kind='kde',
             height=2)

plt.suptitle("Pair Plot of Selected Features Correlated with Win", y=1.02)
plt.show()

This tells some pretty interesting things as well.
It would take a long time to say all of the things that I notice from these graphs so I will just pick what I find the most interesting.

- There is a strong correlation between people who have a positive win streak differential with people who average 2 or more takedowns

- The longest current winstreaks tend to happen when people are in their late 20s - early 30s

- There is a very strong win correlation for people who have a positve AvgTDDif and a current winstreak.

- People under the age of 30 tend to win more than people over the age of 30



Now I want to bin some of my features, I have learned a lot of interesting things about my features and I wnat to put continuous data into better groupings. To do that I had ChatGPT create me a method that when I give it a feaature, and a flag for whether or not it is discrete or continuous, it will make two graphs one with the x axis being the feature and the y axis being teh win percentage and another where the y axis is the count. I will use this to determine what good bins will be for my data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_win_stats(df, feature, discrete=True, bins=10):
    """
    Plots win % and count for a given feature.

    Parameters:
    - df: pandas DataFrame containing the feature and 'Win'
    - feature: column name of the feature to analyze
    - discrete: True if the feature is discrete/categorical, False if continuous
    - bins: number of bins to use if feature is continuous
    """
    if feature not in df.columns:
        print(f"Feature '{feature}' not found in DataFrame.")
        return
    if 'Win' not in df.columns:
        print("Target column 'Win' not found in DataFrame.")
        return

    plot_df = df[[feature, 'Win']].copy()

    if discrete:
        # For discrete features, use the actual values as x-axis
        grouped = plot_df.groupby(feature).agg(
            win_pct=('Win', 'mean'),
            count=('Win', 'count')
        ).reset_index()
        x_labels = grouped[feature].astype(str)
    else:
        # For continuous features, bin the data
        plot_df['bin'] = pd.cut(plot_df[feature], bins=bins)
        grouped = plot_df.groupby('bin').agg(
            win_pct=('Win', 'mean'),
            count=('Win', 'count')
        ).reset_index()
        # Use bin midpoints as x-axis for plotting
        grouped['bin_mid'] = grouped['bin'].apply(lambda x: x.mid)
        x_labels = grouped['bin_mid']

    # Plot Win %
    plt.figure(figsize=(10,4))
    sns.barplot(x=x_labels, y=grouped['win_pct'], color='green')
    plt.ylabel('Win %')
    plt.xlabel(feature if discrete else f"{feature} (binned)")
    plt.title(f"Win % by {feature}")
    plt.xticks(rotation=45)
    plt.ylim(0,1)
    plt.show()

    # Plot Count
    plt.figure(figsize=(10,4))
    sns.barplot(x=x_labels, y=grouped['count'], color='blue')
    plt.ylabel('Count')
    plt.xlabel(feature if discrete else f"{feature} (binned)")
    plt.title(f"Count by {feature}")
    plt.xticks(rotation=45)
    plt.show()


The first one that I want to bin is age.

In [ ]:
plot_win_stats(corr_df_numeric, 'Age', discrete=True)

It looks to me like this calls for 4 bins, the win percentages for agens 18-25 seem similar, and the win percentages from 25-32 seem close, and 32-37, and 37 on. This will help to assign features and stay away from overfitting. I am now going to create these features and assign them to my dataset.



In [ ]:
def binAge(age):
  if age < 25:
    return 1
  elif age < 32:
    return 2
  elif age < 37:
    return 3
  else:
    return 4


corr_df["Age_Bin"] = corr_df.apply(lambda row : binAge(row["Age"]), axis=1)
corr_df.head()


Now that I have binned the Ages, I want to bin height and reach differentials

In [ ]:

plot_win_stats(corr_df_numeric, 'ReachDif', discrete=False, bins=12)



It's pretty obvious here that the reach differential of 178cm is an outlier, and if I had to guess, the opponent's metrics aren't in there, I am going to find this value drop the entry.



In [ ]:
reach_over_100 = corr_df[corr_df['ReachDif'] > 100]

print(reach_over_100)

In [ ]:
print(f"The Size of the DF before dropping {len(corr_df)}")
# corr_df = corr_df[corr_df['ReachDif'] <= 100]

print(f"The Size of the DF after dropping {len(corr_df)}")



The results of that were the following:

The Size of the DF before dropping 11490

The Size of the DF after dropping 6068

This did not make sense so I decided to go back to the the origional df and see what the actual fights were like so that I could clearly see what happened.




In [ ]:
extreme_reach = df[(df['ReachDif'] < -50) | (df['ReachDif'] > 50)][['RedReachCms', 'BlueReachCms', 'ReachDif']]

print(extreme_reach)

The first two seem to be entry errors, they simply didn't take the blue reach into consideration at all, the last one is what I was expecting to be the case for all three.

Since my df is size 11.5k I think it is safe to delete these rows.

In [ ]:
print(corr_df[(corr_df["ReachDif"] > 100)])

In [ ]:
print(f"The Size of the DF before dropping {len(corr_df)}")

index_to_drop = corr_df[corr_df['ReachDif'] >= 70].index
corr_df= corr_df.drop(index_to_drop)


print(f"The Size of the DF after dropping {len(corr_df)}")



print(corr_df[(corr_df["ReachDif"] > 100)])

Lets see if that worked

In [ ]:
corr_df_numeric = corr_df.drop(columns=['Date', 'FinishRoundTime'])

plot_win_stats(corr_df_numeric, 'ReachDif', discrete=False, bins=25)



Having looked at this, there is not a large correlation between winning and reach differential at all, so I don't think there is any reason to bin them based on win%

In [ ]:
plot_win_stats(corr_df_numeric, 'HeightDif', discrete=True)


Tne same is true for Height, they seem very similar for the most part and not very indicative at all of win%.

So there is no point in binning these values. I though about combining them, but since they do not have any correlation with winning. I don't think there is apoint in binning them since they shouldn't really be weighted much if at all.

I could keep on binning all day, but the last thing that I want to bin is odds, I think odds could be very indicative if a fighter will win or not, but the actual raw value

In [ ]:
plot_win_stats(corr_df_numeric, 'Odds', discrete=False, bins = 30)


I am going to bin all odds from -285 and up as 2 since it will be a higher weight for negative odds as negative odds reflect winning. I am going to bin from 0 to -285 as they all have similar win percentages. I will bin odds greater than 200 since they all have efectively no win % and I will bin odds 0 - 100 as they have about the same win percentage

In [ ]:
def binOdds(odds):
  if odds < -285:
    return 2
  elif odds < 0:
    return 1
  elif odds > 200:
    return -1
  else:
    return -2


corr_df["Odds_Binned"] = corr_df.apply(lambda row : binOdds(row["Odds"]), axis=1)
corr_df.head()


#Conclusion
I hae thouroughly enjoyed this project and I think I have learned a lot about my Data, I look forward to out meeting.